# exp_08b Proximity Ablation

In [ ]:
# Install package from GitHub
!pip install git+https://github.com/SilasPignotti/urban-tree-transfer.git -q
# Optional dependencies
!pip install optuna pytorch-tabnet -q

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import drive

drive.mount('/content/drive')

BASE_DIR = Path('/content/drive/MyDrive/dev/urban-tree-transfer')
DATA_DIR = BASE_DIR / 'data'
OUTPUT_DIR = BASE_DIR / 'outputs/phase_3'

(OUTPUT_DIR / 'metadata').mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'figures').mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'logs').mkdir(parents=True, exist_ok=True)

from urban_tree_transfer.config import load_experiment_config
from urban_tree_transfer.experiments import (
    ablation,
    data_loading,
    preprocessing,
    training,
    evaluation,
    visualization,
    models,
    transfer,
    hp_tuning,
)
from urban_tree_transfer.utils import (
    validate_setup_decisions,
    validate_algorithm_comparison,
    validate_hp_tuning_result,
    validate_evaluation_metrics,
    validate_finetuning_curve,
)

In [ ]:
# Load prior CHM decision from exp_08
config = load_experiment_config()
setup_path = OUTPUT_DIR / 'metadata/setup_decisions.json'

if not setup_path.exists():
    raise FileNotFoundError(f"setup_decisions.json not found. Run exp_08 first.")

setup = json.loads(setup_path.read_text())
chm_strategy = setup['chm_strategy']['decision']
chm_features = setup['chm_strategy'].get('included_features', [])

print(f"Loaded CHM decision from exp_08: {chm_strategy}")
print(f"Included CHM features: {chm_features or 'none (Sentinel-2 only)'}")

In [ ]:
# Test both proximity variants with CHM decision applied
variants = config['setup_ablation']['proximity']['variants']

results = []
for variant in variants:
    # Load baseline or filtered dataset
    train_df, val_df, _ = data_loading.load_berlin_splits(
        DATA_DIR / 'phase_2_splits', variant=variant['name']
    )

    # Apply CHM strategy from exp_08
    train_df = ablation.apply_chm_strategy(train_df, chm_strategy)
    val_df = ablation.apply_chm_strategy(val_df, chm_strategy)

    # Get feature columns with CHM decision applied
    feature_cols = data_loading.get_feature_columns(
        train_df,
        include_chm=bool(chm_features),
        chm_features=chm_features or None,
    )

    x_train_scaled, x_val_scaled, _, _ = preprocessing.scale_features(
        train_df[feature_cols], x_val=val_df[feature_cols]
    )

    y_train, label_to_idx, _ = preprocessing.encode_genus_labels(train_df['genus_latin'])
    y_val = val_df['genus_latin'].map(label_to_idx).to_numpy()

    cv = training.create_spatial_block_cv(train_df, n_splits=config['global']['cv_folds'])

    rf = models.create_model('random_forest')
    metrics = training.train_with_cv(
        rf, x_train_scaled, y_train, train_df['block_id'].values, cv
    )

    results.append({
        'variant': variant['name'],
        'val_f1_mean': metrics['val_f1_mean'],
        'val_f1_std': metrics['val_f1_std'],
        'train_val_gap': metrics['train_val_gap'],
        'samples_retained': len(train_df),
    })

results_df = pd.DataFrame(results)
fig_dir = OUTPUT_DIR / 'figures/exp_08b_proximity'
fig_dir.mkdir(parents=True, exist_ok=True)
visualization.plot_ablation_comparison(
    results_df,
    title='Proximity Filter Comparison',
    output_path=fig_dir / 'proximity_ablation.png',
)

print('\nProximity ablation results:')
print(results_df.to_string(index=False))

In [ ]:
# Apply decision rules
rules = config['setup_ablation']['proximity']['decision_rules']

baseline_row = results_df.loc[results_df['variant'] == 'baseline'].iloc[0]
filtered_row = results_df.loc[results_df['variant'] == 'filtered'].iloc[0]

baseline_n = int(baseline_row['samples_retained'])
filtered_n = int(filtered_row['samples_retained'])
sample_loss = 1 - (filtered_n / baseline_n)

baseline_f1 = baseline_row['val_f1_mean']
filtered_f1 = filtered_row['val_f1_mean']
f1_improvement = filtered_f1 - baseline_f1

# Decision logic
if sample_loss > rules['max_sample_loss']:
    decision = 'baseline'
    reasoning = f"Filtered loses {sample_loss:.1%} of samples (>{rules['max_sample_loss']:.1%})"
elif f1_improvement < rules['min_improvement']:
    decision = 'baseline'
    reasoning = f"Improvement {f1_improvement:.3f} < {rules['min_improvement']}"
else:
    decision = 'filtered'
    reasoning = f"Filtered improves F1 by {f1_improvement:.3f} with {sample_loss:.1%} sample loss"

print(f"\nProximity decision: {decision}")
print(f"Reasoning: {reasoning}")
print(f"Sample loss: {sample_loss:.1%} ({baseline_n} → {filtered_n})")
print(f"F1 improvement: {f1_improvement:+.4f}")

In [ ]:
# Update setup_decisions.json
setup['proximity_strategy'] = {
    'decision': decision,
    'reasoning': reasoning,
    'sample_counts': {
        'baseline_n': baseline_n,
        'filtered_n': filtered_n,
        'sample_loss_pct': float(sample_loss),
    },
    'ablation_results': results_df.to_dict(orient='records'),
}

setup_path.write_text(json.dumps(setup, indent=2))
print(f"\nUpdated {setup_path} with proximity decision")

# Validate against schema
validate_setup_decisions(setup_path)
print("Schema validation: PASSED")